Jerry Zhou (zz977)
---
Kaggle Competition (Word + Character TF-IDF + Linear SVM)

**1. Import necessary libraries**

In [1]:
import pandas as pd
import numpy as np
import re
import time
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from scipy.special import expit
from scipy.sparse import hstack
from itertools import product

**2. Set the hyperparameters**

In [2]:
seed = 18
test_split = 0.2

word_ngram_ranges = [(1, 2), (1, 3), (1, 4), (2, 4)]
max_features = 50000
min_freq = 3
max_freq = 1.0
word_lowercase = False

char_analyzer = "char"
char_ngram_ranges = [(2, 5), (3, 6), (4, 7)]
char_max_features = 100000
char_min_freq = 5
char_lowercase = False

word_weights = [0.5, 0.75, 1.0, 1.25, 1.5]

reg_values = [0.001, 0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0]

**3. Load and split the data**

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

X = train["text"].fillna("")
y = train["label"]
test_text = test["text"].fillna("")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_split, random_state=seed, stratify=y)

**4. Convert the text to word and character level TF-IDF features**

In [4]:
def build_vectorizers(word_ngrams, char_ngrams):
    word_vec = TfidfVectorizer(analyzer="word",
                               ngram_range=word_ngrams,
                               max_features=max_features,
                               min_df=min_freq,
                               max_df=max_freq,
                               sublinear_tf=True,
                               lowercase=word_lowercase,
                               token_pattern=r"(?u)\b\w+\b|[^\w\s]")

    char_vec = TfidfVectorizer(analyzer=char_analyzer,
                               ngram_range=char_ngrams,
                               max_features=char_max_features,
                               min_df=char_min_freq,
                               sublinear_tf=True,
                               lowercase=char_lowercase)

    return word_vec, char_vec

**5. Train the SVM and test for the best hyperparameter**

In [5]:
best_auc = 0
best_params = None
results = []

search_start_time = time.perf_counter()

for word_ngrams, char_ngrams, weight in product(word_ngram_ranges, char_ngram_ranges, word_weights):

    print(f"Testing word n-grams={word_ngrams}, char n-grams={char_ngrams}, word weight={weight}")

    word_vec, char_vec = build_vectorizers(word_ngrams, char_ngrams)

    Xw_train = word_vec.fit_transform(X_train)
    Xc_train = char_vec.fit_transform(X_train)
    Xw_val = word_vec.transform(X_val)
    Xc_val = char_vec.transform(X_val)

    X_train_tfidf = hstack([weight * Xw_train, Xc_train]).tocsr()
    X_val_tfidf = hstack([weight * Xw_val, Xc_val]).tocsr()

    for c in reg_values:
        model = LinearSVC(C=c, class_weight="balanced",loss="hinge", max_iter=10000)
        training_start_time = time.perf_counter()
        model.fit(X_train_tfidf, y_train)
        training_time = time.perf_counter() - training_start_time

        train_auc = roc_auc_score(y_train, model.decision_function(X_train_tfidf))
        val_auc = roc_auc_score(y_val, model.decision_function(X_val_tfidf))

        results.append({
            "word_ngrams": word_ngrams,
            "char_ngrams": char_ngrams,
            "word_weight": weight,
            "C": c,
            "train_auc": train_auc,
            "val_auc": val_auc,
            "training_time_seconds": training_time
        })

        print(f"  C={c}: Train AUROC {train_auc:.4f} | Validation AUROC {val_auc:.6f} | Training Time: {training_time:.2f} seconds")

        if val_auc > best_auc:
            best_auc = val_auc
            best_params = {
                "word_ngrams": word_ngrams,
                "char_ngrams": char_ngrams,
                "word_weight": weight,
                "C": c
            }

search_time = time.perf_counter() - search_start_time

print(f"Total hyperparameter search time: {search_time:.2f} seconds")

results_df = pd.DataFrame(results).sort_values(by="val_auc", ascending=False).reset_index(drop=True)

print("\nBest hyperparameters:")
print(best_params)
print("Best validation AUROC:", round(best_auc, 6))

print("\nTop 10 combinations:")
display(results_df.head(10))

Testing word n-grams=(1, 2), char n-grams=(2, 5), word weight=0.5
  C=0.001: Train AUROC 0.9363 | Validation AUROC 0.907946 | Training Time: 0.17 seconds
  C=0.01: Train AUROC 0.9363 | Validation AUROC 0.907946 | Training Time: 0.20 seconds
  C=0.03: Train AUROC 0.9489 | Validation AUROC 0.922104 | Training Time: 0.23 seconds
  C=0.05: Train AUROC 0.9641 | Validation AUROC 0.940029 | Training Time: 0.25 seconds
  C=0.1: Train AUROC 0.9803 | Validation AUROC 0.954489 | Training Time: 0.32 seconds
  C=0.2: Train AUROC 0.9924 | Validation AUROC 0.966110 | Training Time: 0.44 seconds
  C=0.3: Train AUROC 0.9966 | Validation AUROC 0.970718 | Training Time: 0.49 seconds
  C=0.5: Train AUROC 0.9992 | Validation AUROC 0.974949 | Training Time: 0.74 seconds
  C=1.0: Train AUROC 1.0000 | Validation AUROC 0.977005 | Training Time: 1.09 seconds
  C=2.0: Train AUROC 1.0000 | Validation AUROC 0.976017 | Training Time: 1.11 seconds
  C=3.0: Train AUROC 1.0000 | Validation AUROC 0.975690 | Training Ti

,word_ngrams,char_ngrams,word_weight,C,train_auc,val_auc,training_time_seconds
0,"(2, 4)","(2, 5)",1.00,0.5,0.999948,0.980175,0.858969
1,"(2, 4)","(2, 5)",1.00,1.0,1.000000,0.979949,0.902774
2,"(2, 4)","(2, 5)",0.75,1.0,0.999999,0.979944,1.157068
3,"(2, 4)","(2, 5)",1.25,0.5,0.999992,0.979804,0.717622
4,"(2, 4)","(2, 5)",1.00,3.0,1.000000,0.979754,1.049946
5,"(2, 4)","(2, 5)",1.00,5.0,1.000000,0.979754,1.076584
6,"(2, 4)","(2, 5)",1.00,2.0,1.000000,0.979753,1.009532
7,"(1, 4)","(2, 5)",1.00,0.5,0.999946,0.979660,0.977735
8,"(2, 4)","(2, 5)",0.75,2.0,1.000000,0.979607,1.156616
9,"(2, 4)","(2, 5)",0.75,3.0,1.000000,0.979582,1.368186


**6. Train final model** 

In [8]:
final_pipeline_start_time = time.perf_counter()

final_word_vec, final_char_vec = build_vectorizers(best_params["word_ngrams"], best_params["char_ngrams"])

X_full_tfidf = hstack([best_params["word_weight"] * final_word_vec.fit_transform(X),final_char_vec.fit_transform(X)]).tocsr()
X_test_tfidf = hstack([best_params["word_weight"] * final_word_vec.transform(test_text),final_char_vec.transform(test_text)]).tocsr()

final_model = LinearSVC(C=best_params["C"], class_weight="balanced",loss="hinge", max_iter=10000)

final_fit_start_time = time.perf_counter()
final_model.fit(X_full_tfidf, y)
final_fit_time = time.perf_counter() - final_fit_start_time

final_pipeline_time = time.perf_counter() - final_pipeline_start_time

print(f"Final SVM training time: {final_fit_time:.2f} seconds")
print(f"Final training pipeline time (TF-IDF + SVM): {final_pipeline_time:.2f} seconds")

Final SVM training time: 1.48 seconds
Final training pipeline time (TF-IDF + SVM): 41.73 seconds


**7. Make predictions and save results**

In [10]:
test_scores = final_model.decision_function(X_test_tfidf)

test_predictions = expit(test_scores)

submission = pd.DataFrame({
    "id": test["id"],
    "label": np.round(test_predictions, 2)
})

submission.to_csv("tfidf_svm_submission.csv", index=False)
print(submission.shape)
print(submission.head())

(3000, 2)
      id  label
0  13001   0.78
1  13002   0.28
2  13003   0.41
3  13004   0.25
4  13005   0.76
